# ssDNA-Hi-C Analysis Pipeline Tutorial

This notebook runs the complete sshicstuff sample pipeline on one Graal sparse matrix, from input sanity checks to aggregated 4C-like summaries around centromeres and telomeres.
It is a thin, reproducible wrapper around the library entry-points in core/, so every cell maps directly to a function or a CLI command exposed in commands.py.


In [6]:
import os
os.environ['NUMEXPR_MAX_THREADS'] = '4'
os.environ.pop("MPLBACKEND", None)
import pandas as pd

from os.path import join

## 1. Full Pipeline inputs overview

Each run of the pipeline requires the following input files:

Each run of the pipeline requires several input files:

- **`-m / --sparse-matrix`** — Sparse contact matrix produced by *hicstuff* (usually the `graal.tsv` file).  
  Contains pairwise fragment contacts (frag_a, frag_b, contacts).
- **`-c / --oligo-capture`** — CSV/TSV file describing the designed capture oligos (probes).  
  Each probe includes chromosome, start, end, and sequence coordinates. See **design tutorial** to see how to built it. 
- **`-f / --fragments`** — Fragment list generated during genome digestion by *hicstuff*.  
  Provides genomic coordinates of each restriction fragment.
- **`-C / --chr-coord`** — Chromosome coordinate file containing chromosome names, lengths, and centromere/telomere positions.  
  Used for rebinning and aggregation steps.
- **`-a / --additional-groups`** *(optional)* — Probe grouping file allowing the definition of custom probe sets or regions for aggregation.

---

And the following optionnal arguments

- **-b / --binning-sizes** — List of binning sizes for rebinning (e.g. 1000, 10000 bp)
- **-E / --exclude** — Chromosomes to exclude from the analysis
- **-F / --force** — Overwrite existing files
- **-I / --inter** — Keep only inter-chromosomal contacts
- **-n / --flanking-number** — Number of flanking fragments to remove around ssDNA probes (default: 2)
- **-N / --normalize** — Normalize contact frequencies
- **-o / --output** — Output directory
- **-r / --cis-range** — Cis interaction distance in bp (default: 50,000)
- **--binning-aggregate-cen** — Bin size used for centromere aggregation (default: 10 kb)
- **--binning-aggregate-telo** — Bin size used for telomere aggregation (default: 1 kb)
- **--window-size-cen** — Centromere aggregation window (default: 150 kb)
- **--window-size-telo** — Telomere aggregation window (default: 15 kb)
- **--copy-inputs** — Copy all input files to the output directory for reproducibility


In [ ]:
### Common inouts path / filenames
INPUTS_DIR = "../test_data/inputs"
OUTPUTS_DIR = "../test_data/outputs"

graal = "Graal_sample_for_pipeline.txt"
capture = "capture_oligo_for_pipeline.csv"
coords = "chr_coordinates_for_pipeline.tsv"
frags = "fragments_list_for_pipeline.txt"
groups = "probe_groups_for_pipeline.tsv"


## Overview of the pipeline

The notebook calls `pipeline.full_pipeline(...)` which orchestrates the following sequence and writes all artefacts into a run directory named after the sparse matrix.

1. Associate probes to fragments
2. Split reads into dsDNA-only and ssDNA-only matrices
3. Filter contacts to pairs with at least one probe
4. Generate probe-only square profiles
5. Compute 4C-like profiles at 0 kb
6. Rebin profiles to additional resolutions
7. Compute per-probe statistics
8. Aggregate signals near centromeres and telomeres

Each step corresponds to a callable in `core/` and a subcommand in `commands.py`.


In [ ]:
### Full Pipeline
# Uncomment to run
# ARGS = [
#     "-m", join(INPUTS_DIR, graal),
#     "-c", join(INPUTS_DIR, capture),
#     "-C", join(INPUTS_DIR, coords),
#     "-f", join(INPUTS_DIR, frags),
#     "-a", join(INPUTS_DIR, groups),
#     "-n", "2",
#     "-I",
#     "-N",
#     "-o", OUTPUTS_DIR,
#     "-b", "1000",
#     "-b", "2000",
#     "-b", "5000",
#     "-b", "10000",
#     "-r", "50000",
#     "--binning-aggregate-cen", "10000",
#     "--binning-aggregate-telo", "1000",
#     "--window-size-cen", "150000",
#     "--window-size-telo", "15000",
#     "--copy-inputs",
#     "-F",
# ]
#
# cmd = "sshicstuff pipeline " + " ".join(ARGS)
# !{cmd}

## Step-by-step description

### 1. Associate probes to fragments

**What happens:**
Each probe midpoint is matched to its closest fragment (read number) on the same chromosome from the `hicstuff` digest. The probe table gains three new columns: `fragment`, `fragment_start`, and `fragment_end`.

**Output:**
`<oligo_capture>_fragments_associated.csv`

**Notes:**
Input file types are verified, overwrite behavior is controlled by `force`.

In [7]:
## Associate
capture_associated_path = join(OUTPUTS_DIR, "inputs/capture_oligo_for_pipeline_fragments_associated.csv")

ARGS = [
    "associate",
    "-c",  join(INPUTS_DIR, capture),
    "-f",  join(INPUTS_DIR, frags),
    "-o",  capture_associated_path
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
df_associated = pd.read_table(capture_associated_path, sep=",")
df_associated.loc[:, ["name", "fragment", "fragment_start", "fragment_end"]].head()

INFO :: [Associate] : Associate oligo/probe name to fragment/read ID that contains it.
INFO :: [Associate] : Creating a new oligo_capture table : capture_oligo_for_pipeline_fragments_associated.csv
INFO :: [Associate] : oligos associated to fragments successfully.


,name,fragment,fragment_start,fragment_end
0,Native_URA-L-17213-MfeI-RC,74823,0,289
1,Native_URA-L-16220-MfeI-RC,74824,289,655
2,Native_URA-L-15683-SspI-RC,74825,655,1021
3,Native_URA-L-9924-MfeI-RC,74826,1021,1387
4,Native_URA-L-6532-MfeI-RC,74827,1387,1727



### 2. dsDNA-only and ssDNA-only matrices, plus coverage

**What happens:**  
Two sparse matrices are created by selecting contacts involving dsDNA or ssDNA probes from the original sparse one (Graal matrix).\

Optional argument `n_flanking_dsdna` removes neighbors around dsDNA probes.  
Coverage is computed by melting counts per fragment, and fragments crossing bin edges are proportionally split when `bin_size > 0`.
Coverage can be done either on original sparse matrix, or dsDNA or 

**Outputs:**  
`*_dsdna_only.txt`, `*_ssdna_only.txt`, `*_contacts_coverage.bedgraph`, and, if binned, `*_contacts_coverage_<bin>.bedgraph`

**Options:**  
`normalize=True` applies column-sum normalization to coverage values.


In [ ]:
## dsdna only
ARGS = [
    "dsdnaonly",
    "-m", join(INPUTS_DIR, graal),
    "-c", capture_associated_path,
    "-o", join(OUTPUTS_DIR, "Graal_sample_for_pipeline_dsdna_only.tsv"),
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
df_dsdnaonly = pd.read_table(join(OUTPUTS_DIR, "Graal_sample_for_pipeline_dsdna_only.tsv"), sep="\t")
df_dsdnaonly.head()

In [ ]:
## ssdna only
ARGS = [
    "ssdnaonly",
    "-m", join(INPUTS_DIR, graal),
    "-c", capture_associated_path,
    "-o", join(OUTPUTS_DIR, "Graal_sample_for_pipeline_ssdna_only.tsv"),
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
df_ssdnaonly = pd.read_table(join(OUTPUTS_DIR, "Graal_sample_for_pipeline_ssdna_only.tsv"), sep="\t")
df_ssdnaonly.head()

In [ ]:
## Coverage
ARGS = [
    "coverage",
    "-m", join(INPUTS_DIR, graal),
    "-f", join(INPUTS_DIR, frags),
    "--outdir", OUTPUTS_DIR,
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
df_coverage = pd.read_table(join(OUTPUTS_DIR, "Graal_sample_for_pipeline_contacts_coverage.bedgraph"), sep="\t")
df_coverage.head()

In [ ]:
## Coverage with bin sizes
ARGS = [
    "coverage",
    "-m", join(INPUTS_DIR, graal),
    "-f", join(INPUTS_DIR, frags),
    "-c", join(INPUTS_DIR, coords),
    "-b", "5000",
    "--outdir", OUTPUTS_DIR,
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
df_coverage5kb = pd.read_table(join(OUTPUTS_DIR, "Graal_sample_for_pipeline_contacts_coverage_5kb.bedgraph"), sep="\t")
df_coverage5kb.head()

### 3. Filter to probe-bearing pairs

**What happens:**  
Reads the sparse matrix and keeps only contacts where at least one side overlaps a probe fragment.  
Outputs a tidy table with per-side metadata, sorted and deduplicated.

**Output:**  
`*_filtered.tsv`

**Notes:**  
Handles both CSV and TSV inputs, supports artificial chromosomes for ssDNA designs.

In [8]:
## Filter
ARGS = [
    "filter",
    "-m", join(INPUTS_DIR, graal),
    "-c", capture_associated_path,
    "-f", join(INPUTS_DIR, frags),
    "-o", join(OUTPUTS_DIR, "Graal_sample_for_pipeline_filtered.tsv"),
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
filtered_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_filtered.tsv")
df_filtered = pd.read_table(filtered_path, sep="\t")
df_filtered.head()

INFO :: [Filter] : Filtered contacts saved to ../test_data/outputs/Graal_sample_for_pipeline_filtered.tsv


,frag_a,frag_b,contacts,chr_a,start_a,end_a,size_a,gc_content_a,name_a,type_a,sequence_a,chr_b,start_b,end_b,size_b,gc_content_b,name_b,type_b,sequence_b
0,0,8579,1,chr1,0,336,336,0.511905,NaN,NaN,NaN,chr4,64397,64699,302,0.337748,chr4-64420-CDC13,ss,CAACTCACTTGTGGATATCTTCAACAATTTAATAGAAATGAATAGA...
1,0,68254,1,chr1,0,336,336,0.511905,NaN,NaN,NaN,chr15,996325,996743,418,0.397129,chr15-996452-MEK1,ds,ATGAGACCGTTGTATAGCTGCAACCTTGCAACCAAAGATGATATTG...
2,0,74831,2,chr1,0,336,336,0.511905,NaN,NaN,NaN,chr_artificial_dsDNA,2485,2851,366,0.071038,Native_URA-L-3728-SspI-RC,ss,tTCTAATAGTCCTAGGACaCACATGAAGTaCTCATTTGTCAAAtTA...
3,0,74835,1,chr1,0,336,336,0.511905,NaN,NaN,NaN,chr_artificial_dsDNA,3583,3949,366,0.081967,Native_URA-L-1560-SspI-RC,ss,GCTTAGCGCAGCAGTCAAATAgTGAGTAAATTGGTgCGCAACAATG...
4,0,74851,1,chr1,0,336,336,0.511905,NaN,NaN,NaN,chr_artificial_ssDNA,1387,1753,366,0.076503,Probe_URA-L-6532-MfeI-RC,ss,TTCTTGACTTgCTTCTTCTTTGGATaCTACATTTGTGCCAtTTGTA...



### 4. Probe-only square profile

**What happens:**  
Builds a symmetric probe × probe contact matrix restricted to contacts where both sides are probes.  
Useful for QC and probe clustering.

**Output:**  
`probes_matrix.tsv`



### 5. Probe-only square profile

**What happens:**  
Builds a symmetric probe × probe contact matrix restricted to contacts where both sides are probes.  
Useful for QC and probe clustering.

**Output:**  
`probe_to_probe_matrix.tsv`


In [ ]:
## Probe-to-probe matrix + heatmap
probes_matrix_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_probe_to_probe_matrix.tsv")

ARGS = [
    "probe2probe",
    "-c", capture_associated_path,
    "-f", filtered_path,
    "-o", probes_matrix_path,
    "--normalize",
    "-P",
    "--plot-format", "png",
    "--colormap", "YlOrBr",
    "--export-to-cooler",
    "-F",
]

cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
from IPython.display import Image
Image(probes_matrix_path.replace(".tsv", ".png"))

## Reading the exported `.cool` file

The probe-to-probe matrix can also be exported in the `.cool` format for interoperability with Cooler-based tools. In this representation, each probe is encoded as a synthetic bin on an artificial chromosome (`probes`), and pairwise contact values are stored in the pixels table.

Because this matrix is defined in probe space rather than genomic coordinates, probe annotations are stored in a companion file (`*_bins.tsv`), which maps each Cooler `bin_id` to its corresponding probe name and fragment.

In practice:
- the `.cool` file contains the matrix structure and values;
- the `_bins.tsv` file provides the biological annotation of the matrix axes.

Both files should be used together for interpretation and visualization.

In [ ]:
import cooler
import matplotlib.pyplot as plt

cool_path = probes_matrix_path.replace(".tsv", ".cool")
bins_map_path = probes_matrix_path.replace(".tsv", "_bins.tsv")

# Open the cooler file
c = cooler.Cooler(cool_path)

# Load the probe-to-probe matrix
mat = c.matrix(balance=False)[:]

# Load probe annotations
df_bins = pd.read_csv(bins_map_path, sep="\t")
probe_labels = df_bins["probe"].tolist()

print("Cooler bins:")
display(c.bins()[:5])

print("Companion probe annotation table:")
display(df_bins.head())

# Quick visualization with probe labels
plt.figure(figsize=(10, 10))
plt.imshow(mat, cmap="YlOrBr", interpolation="none")
plt.colorbar(label="Normalized contact frequency")

step = max(1, len(probe_labels) // 20)
plt.xticks(range(0, len(probe_labels), step), probe_labels[0::step], rotation=90)
plt.yticks(range(0, len(probe_labels), step), probe_labels[0::step])

plt.title("Probe-to-probe matrix loaded from .cool")
plt.xlabel("Probes")
plt.ylabel("Probes")
plt.tight_layout()
plt.show()

### 5. 4C-like profiles in absolute count

**What happens:**  
For each probe, contacts are grouped by genomic bins defined by fragment coordinates.  
A profile is built as contact counts or normalized frequencies per bin, optionally including additional probe groups.

**Outputs:**  
`*_0kb_profile_contacts.tsv` and, if `normalize=True`, `*_0kb_profile_frequencies.tsv`


In [9]:
## Classical 4C profile no binning, normalized
# First make the profile

# Absolute contacts
profile_0kb_contacts_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_0kb_profile_contacts.tsv")

ARGS = [
    "profile",
    "-c", capture_associated_path,
    "-C", join(INPUTS_DIR, coords),
    "-f", filtered_path,
    "-a", join(INPUTS_DIR, groups),
    "-o", profile_0kb_contacts_path,
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}

# in normalized frequencies
profile_0kb_contacts_norm_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_0kb_profile_frequencies.tsv")
ARGS = [
    "profile",
    "-c", capture_associated_path,
    "-C", join(INPUTS_DIR, coords),
    "-f", filtered_path,
    "-a", join(INPUTS_DIR, groups),
    "-o", profile_0kb_contacts_norm_path,
    "-N",
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}

# Then plot the profile
## Plot4C of this profile

ARGS = [
    "plot4c",
    "-p", profile_0kb_contacts_norm_path,
    "-c", capture_associated_path,
    "-o", OUTPUTS_DIR,
    "-C", join(INPUTS_DIR, coords),
    "-e", "png",
    "-H", "600",
    "-W", "1200",
    "-r", "4",
    "-y", "0",
    "-Y", "0.01",
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}


example_plot4c_probe_path = join(OUTPUTS_DIR, "plots/0kb/raw/Graal_sample_for_pipeline_74823_Native_URA-L-17213-MfeI-RC_frequencies_0kb_.png")
Image(example_plot4c_probe_path)

Traceback (most recent call last):
  File "/home/adminico/miniforge3/envs/sshicstuff_env/bin/sshicstuff", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/adminico/Documents/projects-src/sshicstuff/src/sshicstuff/main.py", line 106, in main
    command.execute()
  File "/home/adminico/Documents/projects-src/sshicstuff/src/sshicstuff/commands.py", line 700, in execute
    plot.plot_profiles(
  File "/home/adminico/Documents/projects-src/sshicstuff/src/sshicstuff/core/plot.py", line 356, in plot_profiles
    b = re.search(r"_(\d+)kb_profile_", profile_contacts_path).group(1)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'group'


NameError: name 'Image' is not defined

### 6. Rebin profiles

**What happens:**  
Converts the unbinned 0 kb profile to new resolutions.  
Fragments spanning multiple bins are proportionally split, and missing bins are zero-filled using a genome-wide template.

**Output:**  
`*_profile_<bin>.tsv`  
The suffix `<bin>` is generated by `get_bin_suffix()`.

In [ ]:
## Rebinning at 1 kb

ARGS = [
    "rebin",
    "-p", profile_0kb_contacts_norm_path,
    "-C", join(INPUTS_DIR, coords),
    "-b", "1000",
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}

profile_1kb_freq_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_1kb_profile_frequencies.tsv")
ARGS = [
    "plot4c",
    "-p", profile_1kb_freq_path,
    "-c", capture_associated_path,
    "-o", OUTPUTS_DIR,
    "-C", join(INPUTS_DIR, coords),
    "-e", "png",
    "-H", "600",
    "-W", "1200",
    "-r", "4",
    "-y", "0",
    "-Y", "0.01",
]

cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}

example_plot4c_probe_1kb_path = join(OUTPUTS_DIR, "plots/1kb/raw/Graal_sample_for_pipeline_74823_Native_URA-L-17213-MfeI-RC_frequencies_1kb_.png")
Image(example_plot4c_probe_1kb_path)

In [ ]:
## Rebinning at 10 kb

ARGS = [
    "rebin",
    "-p", profile_0kb_contacts_norm_path,
    "-C", join(INPUTS_DIR, coords),
    "-b", "10000",
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}

profile_10kb_freq_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_10kb_profile_frequencies.tsv")
ARGS = [
    "plot4c",
    "-p", profile_10kb_freq_path,
    "-c", capture_associated_path,
    "-o", OUTPUTS_DIR,
    "-C", join(INPUTS_DIR, coords),
    "-e", "png",
    "-H", "600",
    "-W", "1200",
    "-r", "4",
    "-y", "0",
    "-Y", "0.03",
]

cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}

example_plot4c_probe_10kb_path = join(OUTPUTS_DIR, "plots/10kb/raw/Graal_sample_for_pipeline_74823_Native_URA-L-17213-MfeI-RC_frequencies_10kb_.png")
Image(example_plot4c_probe_10kb_path)



### 7. Per-probe statistics

**What happens:**  
Computes for each probe:
- total number of contacts  
- coverage relative to all Hi-C contacts  
- cis vs trans fractions (controlled by `cis_range`)  
- chromosome-wise normalized enrichment  
- inter-chromosome-only enrichment  

**Outputs:**  
`*_statistics.tsv`, `*_norm_chr_freq.tsv`, `*_norm_inter_chr_freq.tsv`


In [ ]:
## Get statistics

ARGS = [
    "stats",
    "-m", join(INPUTS_DIR, graal),
    "-p", profile_0kb_contacts_path,
    "-c", capture_associated_path,
    "-C", join(INPUTS_DIR, coords),
    "-o", OUTPUTS_DIR,
    "-r", "50000",
    "-F"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}

stats_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_statistics.tsv")
df_stats = pd.read_csv(stats_path, sep="\t")
df_stats.head()

In [ ]:
norm_chr_freq_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_norm_chr_freq.tsv")
df_norm_chr_freq = pd.read_csv(norm_chr_freq_path, sep="\t")
df_norm_chr_freq.head()

In [ ]:
norm_chr_freq_inter_only_path = join(OUTPUTS_DIR, "Graal_sample_for_pipeline_norm_inter_chr_freq.tsv")
df_norm_chr_freq_inter = pd.read_csv(norm_chr_freq_inter_only_path, sep="\t")
df_norm_chr_freq_inter.head()

### 8. Aggregate near centromeres and telomeres

**What happens:**  
Aggregates a **binned** profile around centromeres and telomeres using window sizes defined by `cen_agg_window_size` and `telo_agg_window_size`.  
Optionally removes intra-chromosomal contacts (`inter_chr_only=True`) and normalizes each probe column.  
Averages are computed per chromosome and across chromosomes.

**Outputs:**  
`<prefix>_mean.tsv`, `<prefix>_median.tsv`, `<prefix>_std.tsv`, and `<prefix>_<fragment>_per_chr.tsv`  
Saved under `aggregated/centromeres/` or `aggregated/telomeres/`.


In [ ]:
## Aggregate on centromeres
ARGS = [
    "aggregate",
    "-p", profile_1kb_freq_path,
    "-c", capture_associated_path,
    "-C", join(INPUTS_DIR, coords),
    "-o", OUTPUTS_DIR,
    "-N",
    "--cen",
    "-w", "150000"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
df_agg_cen = pd.read_table(join(OUTPUTS_DIR, "aggregated/centromeres/Graal_agg_on_cen_norm_mean.tsv"), sep="\t")
df_agg_cen.head()


In [ ]:
df_agg_cen_per_chr = pd.read_table(join(OUTPUTS_DIR, "aggregated/centromeres/Graal_agg_on_cen_norm_avg_ly_per_chr.tsv"), sep="\t")
df_agg_cen_per_chr.head()

In [ ]:
## Aggregate on telomeres
## Aggregate on centromeres
ARGS = [
    "aggregate",
    "-p", profile_1kb_freq_path,
    "-c", capture_associated_path,
    "-C", join(INPUTS_DIR, coords),
    "-o", OUTPUTS_DIR,
    "-N",
    "--tel",
    "-w", "15000"
]
cmd = "sshicstuff " + " ".join(ARGS)
!{cmd}
df_agg_tel = pd.read_table(join(OUTPUTS_DIR, "aggregated/telomeres/Graal_agg_on_telo_norm_mean.tsv"), sep="\t")
df_agg_tel.head()

## Adjustable parameters

- `additional_groups`: Optional table to define probe groups for aggregation  
- `bin_sizes`: Target resolutions for rebinning (e.g., `[1000, 10000]`)  
- `excluded_chr`: Chromosomes to omit from aggregation  
- `cis_region_size`: Size of cis window in base pairs  
- `n_flanking_dsdna`: Number of dsDNA flanking fragments to remove  
- `inter_chr_only`: Exclude intra-chromosome signal during aggregation  
- `normalize`: Apply per-probe normalization for profiles and coverage  
- `cen_agg_window_size`, `telo_agg_window_size`: Window sizes in bp  
- `cen_aggregated_binning`, `telo_agg_binning`: Which binned profile to aggregate  
- `force`, `copy_inputs`: Overwrite control and reproducibility options


## Output folder structure

```bash
<run_dir>/
  inputs/
    sample.txt
    oligos.csv
    fragments.txt
    chr_coords.tsv

  <sample>_dsdna_only.txt
  <sample>_ssdna_only.txt
  <sample>_filtered.tsv
  <sample>_contacts_coverage.bedgraph
  <sample>_contacts_coverage_<bin>.bedgraph
  <sample>_0kb_profile_contacts.tsv
  <sample>_0kb_profile_frequencies.tsv
  <sample>_profile_<bin>.tsv
  <sample>_statistics.tsv
  <sample>_norm_chr_freq.tsv
  <sample>_norm_inter_chr_freq.tsv

  aggregated/
    centromeres/
      *_mean.tsv
      *_median.tsv
      *_std.tsv
    telomeres/
      *_mean.tsv
      *_median.tsv
      *_std.tsv
